# Notebook 04 – Community Detection

**DigitAfrica Workshop 2026 – Identifying Epistemic Enclaves and Understanding Polarisation**

## Learning objectives
- Apply standard community-detection algorithms to the interaction network
- Compare partitions using modularity and NMI
- Relate detected communities to known group labels

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# cdlib provides a unified interface for community detection
import cdlib
from cdlib import algorithms

%matplotlib inline

## 1. Reconstruct the network

> Reuse the synthetic homophilic network from Notebook 03, or load your own graph.

In [ ]:
rng = np.random.default_rng(1)
n_nodes = 100
groups = {i: ("A" if i < 50 else "B") for i in range(n_nodes)}

G = nx.Graph()
G.add_nodes_from(groups.keys())
nx.set_node_attributes(G, groups, "group")

for u in range(n_nodes):
    for v in range(u + 1, n_nodes):
        same_group = groups[u] == groups[v]
        prob = 0.08 if same_group else 0.01
        if rng.random() < prob:
            G.add_edge(u, v)

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

## 2. Run community detection

In [ ]:
# Louvain method
louvain = algorithms.louvain(G, seed=42)
print(f"Louvain: {len(louvain.communities)} communities")
print(f"Modularity: {louvain.newman_girvan_modularity().score:.4f}")

## 3. Visualise communities

In [ ]:
palette = plt.cm.tab10.colors
node_community = {}
for cid, comm in enumerate(louvain.communities):
    for node in comm:
        node_community[node] = cid

color_map = [palette[node_community[n] % len(palette)] for n in G.nodes()]
pos = nx.spring_layout(G, seed=42)

fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx(
    G, pos=pos, node_color=color_map,
    node_size=60, with_labels=False,
    edge_color="grey", alpha=0.7, ax=ax
)
ax.set_title("Detected communities (Louvain)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Compare to ground-truth labels

In [ ]:
from cdlib import evaluation

ground_truth = cdlib.NodeClustering(
    communities=[list(range(50)), list(range(50, 100))],
    graph=G,
    method_name="ground_truth"
)

nmi = evaluation.normalized_mutual_information(louvain, ground_truth)
print(f"NMI (Louvain vs ground truth): {nmi.score:.4f}")